# 01 · Calidad de datos y hallazgos fijados (PLAN §2)

Cada sección reproduce un hallazgo medido del plan y lo **verifica con `assert`**: si los datos cambian, el notebook falla en lugar de dejar un plan desactualizado. Requiere `data/raw/` (ignorada por git).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.etl.load import load_migration, load_occupancy, load_survey
from src.etl.clean import clean_migration, clean_occupancy, clean_survey
from src.config import SURVEY_BORDER_MAP, IZABAL_BORDERS

pd.set_option("display.width", 140)

In [ ]:
mig = clean_migration(load_migration())
occ = clean_occupancy(load_occupancy())
sur = clean_survey(load_survey())
print({"migración": mig.shape, "ocupación": occ.shape, "encuesta": sur.shape})

## 2.1 La capacidad instalada no existe en las fuentes
El CSV de ocupación solo trae 4 columnas; el Módulo 3 pide establecimientos/habitaciones/plazas cama.

In [ ]:
import src.config as cfg
cols = pd.read_csv(cfg.OCCUPANCY_FILE, nrows=0).columns.tolist()
print(cols)
assert cols == ["anio", "mes", "departamento", "indice_ocupacion"]
print(occ.shape, occ["departamento"].nunique(), "departamentos")
assert occ.shape[0] == 792 and occ["departamento"].nunique() == 22
assert occ["fecha"].min() == pd.Timestamp("2023-01-01") and occ["fecha"].max() == pd.Timestamp("2025-12-01")

## 2.2 Discontinuidad marítima de Izabal

In [ ]:
mar = mig[mig["Frontera"].isin(IZABAL_BORDERS + ["04 Santo Tomás de Castilla"])]
vis = mar[mar["es_visita"]]
resumen = vis.groupby("Frontera").agg(desde=("fecha", "min"), hasta=("fecha", "max"), visitantes=("Viajero", "sum"))
display(resumen)
anio = mar.assign(a=mar["fecha"].dt.year).groupby(["a", "Tipo de Viajero"])["Viajero"].sum().unstack()
display(anio)
assert mar[mar["fecha"].dt.year == 2022].empty, "2022 debería tener 0 filas en las 3 fronteras"

## 2.3 La encuesta es la única serie continua de Izabal

In [ ]:
iz = sur[sur["es_izabal"]]
m = iz.groupby(iz["fecha"].dt.to_period("M")).size()
full = pd.period_range(m.index.min(), m.index.max(), freq="M")
huecos = [str(p) for p in full if p not in m.index]
print(len(iz), "registros;", m.index.min(), "→", m.index.max(), "; huecos:", huecos)
assert huecos == ["2025-01"]
print("registros/mes: min", m.min(), "max", m.max())

## 2.4 Los bordes de la encuesta mapean 1:1 con migración

In [ ]:
from src.utils.helpers import mapa_frontera
fr = sur["frontera_nombre"].dropna().unique()
mapeo = pd.DataFrame({"encuesta": fr}).assign(migracion=lambda d: mapa_frontera(d["encuesta"]))
en_mig = set(mig["Frontera"].unique())
mapeo["existe_en_migracion"] = mapeo["migracion"].isin(en_mig)
display(mapeo)
mapeados = mapeo.dropna(subset=["migracion"])
assert len(SURVEY_BORDER_MAP) == 11 and mapeados["existe_en_migracion"].all()
print("Mapeadas:", len(mapeados), "· sin mapeo:", mapeo["migracion"].isna().sum())

In [ ]:
# Fronteras mapeadas con visitantes continuos 2009-2026 (candidatas a ISABAL_FEEDER_BORDERS)
v = mig[mig["es_visita"] & mig["Frontera"].isin(SURVEY_BORDER_MAP.values())]
meses = v.groupby("Frontera")["fecha"].nunique()
continuas = meses[meses == pd.date_range("2009-01-01", "2026-06-01", freq="MS").size]
print(len(continuas), "con serie continua:"); print(sorted(continuas.index))

## 2.5 La ocupación de Izabal está completa (36 meses)

In [ ]:
oz = occ[occ["departamento"] == "Izabal"].set_index("fecha")["indice_ocupacion"]
assert len(oz) == 36 and oz.notna().all()
perfil = (oz.groupby(oz.index.month).mean() / oz.mean()).round(2)
print(perfil.to_dict())
assert perfil.idxmin() == 10 and perfil.idxmax() == 4

## 2.6 Calidad: mojibake y valores fuera de rango (decisiones de limpieza)

In [ ]:
for c in ["País", "Región", "Frontera"]:
    malos = mig[c][mig[c].str.contains("Ã", na=False)].unique()
    print(c, "→ valores con mojibake:", malos[:10])
print("noches > 365 anuladas:", int((sur["noches"] > 365).sum()))
assert occ["indice_ocupacion"].between(0, 1).all()